# 05 · Auditoría de legibilidad y OCR piloto — Memoria Chilena

**Objetivo:** medir la cobertura real del corpus y preparar una comparación controlada de OCR **antes** de construir los embeddings del GraphRAG.

Este notebook es compatible con **Jupyter local**, usa automáticamente `data/output/etapa4.zip`, **no vuelve a descargar documentos** y genera `data/output/etapa5.zip`. No es necesario ejecutar de nuevo las etapas 1–4.

El OCR piloto se ejecuta solamente **si tienes los PDF locales y Tesseract instalado y disponible**. Si no está disponible, no aparece un error: se produce la auditoría de texto y se registra el OCR como pendiente. No se descargan herramientas externas de forma oculta.

**Importante:** los PDF de 600+ MB en total **no vienen dentro del ZIP del proyecto**. Si ya los descargaste en `data/pdf/`, conserva esa carpeta al actualizar el repositorio.

In [1]:
from pathlib import Path
import io, json, re, csv, shutil, subprocess, zipfile, math, random
from datetime import datetime, timezone
import numpy as np
import pandas as pd

CWD=Path.cwd().resolve()
ROOT=CWD.parent if CWD.name.lower()=='notebooks' else CWD
DATA=ROOT/'data'
assert DATA.is_dir(),f'No existe data/: {DATA}'
ARCHIVO=DATA/'output'/'etapa4.zip'
if not ARCHIVO.exists(): ARCHIVO=DATA/'input'/'etapa4.zip'
assert ARCHIVO.exists(),f'Falta etapa4.zip en data/output o data/input: {ARCHIVO}'
SALIDA=DATA/'output'/'etapa5'
SALIDA.mkdir(parents=True,exist_ok=True)
PDFS=DATA/'pdf'
print('Raíz del proyecto:',ROOT)
print('Entrada:',ARCHIVO)
print('Salida:',SALIDA)

Raíz del proyecto: C:\Users\jvera\Documents\memoria_chilena
Entrada: C:\Users\jvera\Documents\memoria_chilena\data\output\etapa4.zip
Salida: C:\Users\jvera\Documents\memoria_chilena\data\output\etapa5


In [2]:
with zipfile.ZipFile(ARCHIVO) as z:
    obligatorios={'paginas_texto.csv','fragmentos.csv','registro_descargas.csv','informe_etapa4.json'}
    faltantes=obligatorios-set(z.namelist())
    assert not faltantes,f'Faltan archivos en etapa4.zip: {faltantes}'
    paginas=pd.read_csv(io.BytesIO(z.read('paginas_texto.csv'))).fillna({'texto':''})
    trozos=pd.read_csv(io.BytesIO(z.read('fragmentos.csv')))
    descargas=pd.read_csv(io.BytesIO(z.read('registro_descargas.csv')))
    informe=json.loads(z.read('informe_etapa4.json').decode('utf-8'))
assert len(paginas)==informe['paginas_extraidas']
assert len(trozos)==informe['fragmentos_extraidos']
assert set(paginas['mcid'])<=set(descargas['mcid'])
print('Páginas:',len(paginas),'| Fragmentos provisionales:',len(trozos),'| Documentos:',paginas.mcid.nunique())

Páginas: 2641 | Fragmentos provisionales: 4286 | Documentos: 6


In [3]:
# Señales cuantitativas de texto posiblemente deteriorado.
# Nunca se presentan como una medición validada de CER/WER ni como sello de legibilidad.
def medir_texto(s):
    s=str(s or '')
    letras=sum(c.isalpha() for c in s)
    simbolos=sum(not(c.isalnum() or c.isspace()) for c in s)
    tokens=re.findall(r'[^\W\d_]+',s,flags=re.UNICODE)
    cortos=sum(len(t)<=2 for t in tokens)
    largos=sum(len(t)>=22 for t in tokens)
    return pd.Series({'caracteres_medidos':len(s),
      'proporcion_letras':round(letras/max(1,len(s)),3),
      'proporcion_simbolos':round(simbolos/max(1,len(s)),3),
      'proporcion_tokens_cortos':round(cortos/max(1,len(tokens)),3),
      'proporcion_tokens_largos':round(largos/max(1,len(tokens)),3),
      'numero_tokens':len(tokens)})

metricas=paginas['texto'].apply(medir_texto)
audit=pd.concat([paginas.drop(columns=['texto']).reset_index(drop=True),metricas.reset_index(drop=True)],axis=1)
# Se usa la longitud SOLO para generar prioridades, nunca para afirmar que una página está correctamente transcrita.
def clasificar(r):
    if r.caracteres_medidos<80: return 'sin_texto_suficiente'
    if (r.proporcion_simbolos>.13 or r.proporcion_letras<.56 or
        r.proporcion_tokens_cortos>.49 or r.proporcion_tokens_largos>.12):
        return 'texto_con_senales_de_ruido'
    return 'texto_pendiente_validacion'
audit['diagnostico_automatico']=audit.apply(clasificar,axis=1)
audit['texto_aprobado_para_rag']=False
resumen=(audit.groupby('mcid').agg(paginas=('pagina_pdf','count'),
            sin_texto=('diagnostico_automatico',lambda s:(s=='sin_texto_suficiente').sum()),
            senales_ruido=('diagnostico_automatico',lambda s:(s=='texto_con_senales_de_ruido').sum()),
            pendientes_validacion=('diagnostico_automatico',lambda s:(s=='texto_pendiente_validacion').sum()),
            paginas_con_80=('caracteres_medidos',lambda s:(s>=80).sum()),
            caracteres_mediana=('caracteres_medidos','median')).reset_index())
resumen['porcentaje_80']=((resumen.paginas_con_80/resumen.paginas)*100).round(1)
audit.to_csv(SALIDA/'diagnostico_paginas.csv',index=False,encoding='utf-8-sig')
resumen.to_csv(SALIDA/'resumen_por_documento.csv',index=False,encoding='utf-8-sig')
display(resumen)
print('Advertencia: texto_pendiente_validacion no significa texto legible o correcto.')

,mcid,paginas,sin_texto,senales_ruido,pendientes_validacion,paginas_con_80,caracteres_mediana,porcentaje_80
0,MC0014753,181,123,45,13,58,5.0,32.0
1,MC0014754,287,234,8,45,53,1.0,18.5
2,MC0014755,545,73,12,460,472,2814.0,86.6
3,MC0014757,570,76,12,482,494,1850.0,86.7
4,MC0014760,157,80,10,67,77,47.0,49.0
5,MC0014761,901,329,472,100,572,1317.0,63.5


Advertencia: texto_pendiente_validacion no significa texto legible o correcto.


In [4]:
# Muestra reproducible y estratificada: 4 páginas por documento (si hay suficientes).
# Ninguna página queda aprobada sin revisión humana.
rng=random.Random(20261109)
seleccion=[]
for mcid,grupo in audit.groupby('mcid'):
    grupo=grupo.sort_values('pagina_pdf')
    utilizadas=set()
    for categoria in ['sin_texto_suficiente','texto_con_senales_de_ruido','texto_pendiente_validacion']:
        candidatos=grupo[(grupo.diagnostico_automatico==categoria) & (~grupo.pagina_pdf.isin(utilizadas))]
        if not candidatos.empty:
            # Cuantil central de páginas para no seleccionar siempre cubiertas/prólogos.
            elegido=candidatos.iloc[len(candidatos)//2]
            utilizadas.add(int(elegido.pagina_pdf))
            seleccion.append((mcid,int(elegido.pagina_pdf),categoria))
    # Revisión adicional en el tercio final de cada obra.
    candidatos=grupo[~grupo.pagina_pdf.isin(utilizadas)]
    if not candidatos.empty:
        elegido=candidatos.iloc[min(len(candidatos)-1,int(len(candidatos)*.75))]
        seleccion.append((mcid,int(elegido.pagina_pdf),'muestra_adicional'))
        utilizadas.add(int(elegido.pagina_pdf))

muestra=pd.DataFrame(seleccion,columns=['mcid','pagina_pdf','criterio_muestreo'])
muestra=muestra.merge(paginas[['mcid','pagina_pdf','texto','caracteres','ficha','fuente_pdf']],on=['mcid','pagina_pdf'],how='left')
muestra['extracto']=muestra['texto'].fillna('').str.slice(0,650)
muestra['evaluacion_humana']=''
muestra['observaciones_humanas']=''
muestra.drop(columns=['texto']).to_csv(SALIDA/'muestra_revision_manual.csv',index=False,encoding='utf-8-sig')
print('Páginas seleccionadas:',len(muestra))
display(muestra[['mcid','pagina_pdf','criterio_muestreo','caracteres']])

Páginas seleccionadas: 24


,mcid,pagina_pdf,criterio_muestreo,caracteres
0,MC0014753,77,sin_texto_suficiente,5
1,MC0014753,104,texto_con_senales_de_ruido,710
2,MC0014753,140,texto_pendiente_validacion,274
3,MC0014753,136,muestra_adicional,289
4,MC0014754,141,sin_texto_suficiente,8
5,MC0014754,146,texto_con_senales_de_ruido,88
6,MC0014754,156,texto_pendiente_validacion,887
7,MC0014754,217,muestra_adicional,29
8,MC0014755,149,sin_texto_suficiente,0
9,MC0014755,213,texto_con_senales_de_ruido,1787


## Comparación visual (opcional, sin instalación adicional)

Si mantuviste los PDF dentro de `data/pdf/`, se guardarán imágenes de las páginas seleccionadas en `data/output/etapa5/vistas/` para compararlas con el texto. Esas imágenes **no se incluyen en el ZIP** y no deben subirse a GitHub. La ausencia de PDF no bloquea el notebook.

In [5]:
# Exportar imágenes reducidas de la muestra; solo a disco local.
try:
    import fitz
    F_PDF=True
except ImportError:
    F_PDF=False
vistas=SALIDA/'vistas'
vistas.mkdir(parents=True,exist_ok=True)
renderizadas=0
if F_PDF:
    for mcid,grupo in muestra.groupby('mcid'):
        p=PDFS/f'{mcid}.pdf'
        if not p.exists(): continue
        try:
            with fitz.open(p) as doc:
                for numero in grupo['pagina_pdf']:
                    if not 1<=numero<=len(doc): continue
                    pagina=doc[int(numero)-1]
                    scale=min(1.4,900/max(1,pagina.rect.width))
                    pix=pagina.get_pixmap(matrix=fitz.Matrix(scale,scale),alpha=False)
                    pix.save(str(vistas/f'{mcid}_p{int(numero):04d}.png'))
                    renderizadas+=1
        except Exception as exc:
            print('Problema al renderizar',mcid,str(exc)[:120])
print('Vistas de página locales:',renderizadas)

C:\Users\jvera\anaconda3\Lib\site-packages\fitz\__init__.py:17: FutureWarning: The `fitz` API is deprecated and will be removed in future. Use `import pymupdf` instead.
  warnings.warn(


Vistas de página locales: 24


## OCR piloto (opcional y seguro)

Solo se intentará si Jupyter encuentra `tesseract` (instalado por separado) y los PDF ya están disponibles. No se instalará nada automáticamente. Se realiza un **experimento pequeño**, nunca OCR masivo. Los idiomas configurados deben existir en tu Tesseract; si `spa` no está disponible, se registra la ausencia y se omite el piloto. Las lenguas indígenas y la tipografía histórica requieren evaluación específica.

In [6]:
# Importante: resultados del OCR piloto no se incorporan todavía a fragmentos ni a embeddings.
ocr_rows=[]
idiomas=[]
tesseract=shutil.which('tesseract')
if tesseract:
    probe=subprocess.run([tesseract,'--list-langs'],capture_output=True,text=True,timeout=20)
    idiomas=[x.strip() for x in probe.stdout.splitlines() if x.strip() and 'available languages' not in x.lower()]
if not tesseract or not F_PDF or 'spa' not in idiomas:
    print('OCR piloto omitido: Tesseract/PyMuPDF/español no disponibles. La auditoría continúa normalmente.')
else:
    from tempfile import TemporaryDirectory
    # Dos páginas por documento: como máximo 12 para evitar horas de procesamiento involuntario.
    try:
        from PIL import Image
    except ImportError:
        Image=None
    for mcid,grupo in muestra.groupby('mcid'):
        path=PDFS/f'{mcid}.pdf'
        if not path.exists(): continue
        if Image is None:
            print('Pillow no disponible; OCR piloto omitido.'); break
        try:
            with fitz.open(path) as doc:
                for _,r in grupo.head(2).iterrows():
                    numero=int(r.pagina_pdf)
                    if numero<1 or numero>len(doc):continue
                    pix=doc[numero-1].get_pixmap(matrix=fitz.Matrix(2.0,2.0),alpha=False)
                    with TemporaryDirectory() as tmp:
                        entrada=Path(tmp)/'pagina.png';salida=Path(tmp)/'ocr'
                        pix.save(str(entrada))
                        proc=subprocess.run([tesseract,str(entrada),str(salida),'-l','spa'],capture_output=True,text=True,timeout=120)
                        texto=(salida.with_suffix('.txt').read_text(encoding='utf-8',errors='replace')
                            if proc.returncode==0 and salida.with_suffix('.txt').exists() else '')
                        ocr_rows.append({'mcid':mcid,'pagina_pdf':numero,
                          'texto_nativo_caracteres':len(str(r.texto or '')),
                          'texto_ocr_caracteres':len(texto),
                          'extracto_ocr':texto[:800],
                          'estado':'ocr_ok' if texto else f'ocr_sin_texto_codigo_{proc.returncode}',
                          'evaluacion_humana':''})
        except Exception as exc:
            print('OCR piloto falló en',mcid,repr(exc)[:150])
ocr_cols=['mcid','pagina_pdf','texto_nativo_caracteres','texto_ocr_caracteres',
          'extracto_ocr','estado','evaluacion_humana']
ocr_df=pd.DataFrame(ocr_rows,columns=ocr_cols)
ocr_df.to_csv(SALIDA/'comparacion_ocr_piloto.csv',index=False,encoding='utf-8-sig')
print('Páginas con OCR piloto:',len(ocr_df))

OCR piloto omitido: Tesseract/PyMuPDF/español no disponibles. La auditoría continúa normalmente.
Páginas con OCR piloto: 0


In [7]:
# Paquete de auditoría reproducible: NO incluye PDFs, imágenes de páginas ni textos completos.
resultados={
 'fecha_utc':datetime.now(timezone.utc).isoformat(),
 'etapa4_sha256':__import__('hashlib').sha256(ARCHIVO.read_bytes()).hexdigest(),
 'documentos':int(paginas.mcid.nunique()),
 'paginas':int(len(paginas)),
 'paginas_con_80_caracteres':int((audit.caracteres_medidos>=80).sum()),
 'paginas_con_menos_de_80':int((audit.caracteres_medidos<80).sum()),
 'paginas_con_senales_de_ruido':int((audit.diagnostico_automatico=='texto_con_senales_de_ruido').sum()),
 'fragmentos_provisionales':int(len(trozos)),
 'paginas_muestreadas':int(len(muestra)),
 'imagenes_locales_generadas':int(renderizadas),
 'paginas_con_ocr_piloto':int(len(ocr_df)),
 'observaciones':['No hay validación humana; diagnóstico no equivale a CER/WER',
                  'No se autoriza indexación histórica automática a partir de este informe',
                  'Los números de página refieren al visor PDF, no necesariamente a la paginación impresa',
                  'Tesseract es solo una línea base; puede rendir mal con tipografía colonial y lenguas indígenas',
                  'El ZIP omite PDF e imágenes para no redistribuir archivos del portal']
}
(SALIDA/'informe_etapa5.json').write_text(json.dumps(resultados,ensure_ascii=False,indent=2),encoding='utf-8')
archivofinal=DATA/'output'/'etapa5.zip'
with zipfile.ZipFile(archivofinal,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for item in sorted(SALIDA.iterdir()):
        if item.is_file(): z.write(item,arcname=item.name)
assert archivofinal.exists()
print(json.dumps(resultados,ensure_ascii=False,indent=2))
print('ZIP generado:',archivofinal)

{
  "fecha_utc": "2026-11-09T18:35:58.292704+00:00",
  "etapa4_sha256": "ccb5140dbfd3e682b6cbe77357e389314f9fc84ad983711959c2c96f931b9e1f",
  "documentos": 6,
  "paginas": 2641,
  "paginas_con_80_caracteres": 1726,
  "paginas_con_menos_de_80": 915,
  "paginas_con_senales_de_ruido": 559,
  "fragmentos_provisionales": 4286,
  "paginas_muestreadas": 24,
  "imagenes_locales_generadas": 24,
  "paginas_con_ocr_piloto": 0,
  "observaciones": [
    "No hay validación humana; diagnóstico no equivale a CER/WER",
    "No se autoriza indexación histórica automática a partir de este informe",
    "Los números de página refieren al visor PDF, no necesariamente a la paginación impresa",
    "Tesseract es solo una línea base; puede rendir mal con tipografía colonial y lenguas indígenas",
    "El ZIP omite PDF e imágenes para no redistribuir archivos del portal"
  ]
}
ZIP generado: C:\Users\jvera\Documents\memoria_chilena\data\output\etapa5.zip
